In [46]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix
import plotly.express as px

In [47]:
orders = pd.read_csv("../data/cleaned/Orders.csv")
orders.head()

,order_id,customer_id,order_date,payment_method_id,status,grand_total
0,100001290,19328,2016-11-23,4,complete,332.0
1,100002382,40590,2017-04-10,4,canceled,1149.0
2,100147443,1,2016-07-01,4,complete,1950.0
3,100147444,2,2016-07-01,4,canceled,240.0
4,100147445,3,2016-07-01,4,canceled,2450.0


In [48]:
orders.head()

,order_id,customer_id,order_date,payment_method_id,status,grand_total
0,100001290,19328,2016-11-23,4,complete,332.0
1,100002382,40590,2017-04-10,4,canceled,1149.0
2,100147443,1,2016-07-01,4,complete,1950.0
3,100147444,2,2016-07-01,4,canceled,240.0
4,100147445,3,2016-07-01,4,canceled,2450.0


In [49]:
selected = orders["status"].isin(["canceled", "complete"])
data = orders[selected].copy()

data["is_canceled"] = (data["status"] == "canceled").astype(int)

data[["status", "is_canceled"]].head()
data.head()

,order_id,customer_id,order_date,payment_method_id,status,grand_total,is_canceled
0,100001290,19328,2016-11-23,4,complete,332.0,0
1,100002382,40590,2017-04-10,4,canceled,1149.0,1
2,100147443,1,2016-07-01,4,complete,1950.0,0
3,100147444,2,2016-07-01,4,canceled,240.0,1
4,100147445,3,2016-07-01,4,canceled,2450.0,1


In [50]:
data["order_date"] = pd.to_datetime(data["order_date"])
data["order_weekday"] = data["order_date"].dt.day_name()

data[["order_date", "order_weekday"]].head()

,order_date,order_weekday
0,2016-11-23,Wednesday
1,2017-04-10,Monday
2,2016-07-01,Friday
3,2016-07-01,Friday
4,2016-07-01,Friday


In [51]:
X = data[["payment_method_id", "order_weekday"]]
y = data["is_canceled"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (318158, 2)
y shape: (318158,)


 ## i want to split the data based on it's date so i sort it ##
 - oldest for train 
 - newest for testing

In [52]:
dates = data["order_date"].sort_values()
split_date = dates.iloc[int(len(dates) * 0.8)]

train_rows = data["order_date"] < split_date
test_rows = data["order_date"] >= split_date

X_train = X[train_rows]
y_train = y[train_rows]

X_test = X[test_rows]
y_test = y[test_rows]

print("Test starts on:", split_date.date())
print("Training orders:", len(X_train))
print("Test orders:", len(X_test))

Test starts on: 2018-02-19
Training orders: 252083
Test orders: 66075


## encoding

In [53]:
encoder = OneHotEncoder(handle_unknown="ignore")

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

print("Before encoding:", X_train.shape)
print("After encoding:", X_train_encoded.shape)

Before encoding: (252083, 2)
After encoding: (252083, 24)


## فاث ةخيثم

In [54]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_encoded, y_train)
predictions = model.predict(X_test_encoded)
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)

In [55]:
print("Accuracy:", f"{accuracy * 100:.2f}%")
print("Precision:", f"{precision * 100:.2f}%")
print("Recall:", f"{recall * 100:.2f}%")
print("F1 Score:", f"{f1 * 100:.2f}%")

Accuracy: 77.69%
Precision: 82.70%
Recall: 87.11%
F1 Score: 84.85%


In [56]:
fig = px.imshow(
    matrix,
    x=["Predicted complete", "Predicted canceled"],
    y=["Actual complete", "Actual canceled"],
    text_auto=",.0f",
    color_continuous_scale="Blues",
    labels={"x": "Predicted status", "y": "Actual status", "color": "Orders"},
    title="Order Cancellation Confusion Matrix"
)

fig.show()